In [ ]:
from netgen.meshing import Mesh as NGMesh, MeshPoint, Element1D, Element0D, Pnt
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw, FieldLines, AddFieldLines

import matplotlib.pylab as plt
import numpy as np

In [ ]:
def CapacitorGeometry():

    air = MoveTo(0, 0).RectangleC(30, 30).Face()
    air.edges.name = "Outer"
    air.faces.name = "air"

    el_u = MoveTo(0, 1).RectangleC(5, 0.5).Face()
    el_u.edges.name = "el_u"
    el_u.faces.name = "el_u"

    el_d = MoveTo(0, -1).RectangleC(5, 0.5).Face()
    el_d.edges.name = "el_d"
    el_d.faces.name = "el_d"

    dielectric = MoveTo(0, 0).RectangleC(4, 1.5).Face()
    dielectric.faces.name = "dielectric"

    shape = Glue([air - dielectric, dielectric])
    shape = shape - el_u - el_d

    shape.edges["el.*"].maxh=0.2
    shape.vertices["el.*"].maxh=0.2
    
    return shape


def CapacitorMesh(shape, h_max):
    
    mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=h_max))

    return mesh


def CapacitorSolver(mesh, FE_order, epsr):

    fes_potential = H1(mesh, order=FE_order, dirichlet="el.*")

    u = fes_potential.TrialFunction()
    v = fes_potential.TestFunction()

    gfu = GridFunction(fes_potential)
    gfu.Interpolate(mesh.BoundaryCF({"el_u":1, "el_d":-1 }), mesh.Boundaries(".*"))

    a = BilinearForm(epsr*grad(u)*grad(v)*dx).Assemble()
    
    inv = a.mat.Inverse(freedofs=fes_potential.FreeDofs())
    gfu.vec.data -= inv@a.mat * gfu.vec

    return gfu


def CapacitorErrorEstimator(mesh, gf_phi):

    fes_flux = HDiv(mesh, order=FE_order-1)

    gf_E = GridFunction(fes_flux)
    E = -epsr*grad(gf_phi)

    gf_E.Set(E)

    gf_error = 1/epsr*(E-gf_E)*(E-gf_E)
    ZZ_error = Integrate(gf_error, mesh, VOL, element_wise=True)

    return gf_error, ZZ_error



In [ ]:
h_max = 1
geo = CapacitorGeometry()
mesh = CapacitorMesh(geo, h_max)

epsr_air, epsr_dielectric = 1.0, 2.0
epsr = mesh.MaterialCF({"air": epsr_air, "dielectric": epsr_dielectric})

FE_order = 3

gf_phi = CapacitorSolver(mesh, FE_order, epsr)

In [ ]:
gf_error, ZZ_error = CapacitorErrorEstimator(mesh, gf_phi)
Draw(gf_error, mesh)

In [ ]:
print(np.array(ZZ_error)[:10], "...")

In [ ]:
maxerr = max(ZZ_error)
print ("maxerr = ", maxerr)